<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c03-header.png" alt="Nextia Learning · Essential Mathematics and Statistics for AI" width="100%">

# Solution: Compare alternatives

**[Compare alternatives](https://learning.nextia-ai.com/courses/math/m05/compare-alternatives/)** · Essential Mathematics and Statistics for AI · Module 5, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**Optional: check the numbers in Python.** This notebook is optional. The lesson works without code: a calculator is enough. Use the notebook to check the lesson's numbers and to redo its worked examples and tasks in Python.

**You will** compare model A and model B on the same 300 tickets: the effect size, the tickets where they disagree, a coin-flip test with its p-value, and a 95% interval for the difference. Then check a before-and-after claim, and do the module practice with model C.

| | |
|---|---|
| **Time** | About 25 minutes, after you read the lesson |
| **Needs** | Nothing to install and no account: only Python's standard library. It runs in Colab, Kaggle or any Jupyter. |
| **You should know** | The 95% interval for an accuracy, √(p(1 − p)/n), from [Estimate uncertainty: Worked example 2](https://learning.nextia-ai.com/courses/math/m05/estimate-uncertainty/#worked-example-2-the-accuracy-of-model-b). The binomial distribution (coin flips), from [Distributions in practice: Binomial](https://learning.nextia-ai.com/courses/math/m03/distributions-in-practice/#binomial-urgent-tickets-in-a-batch-of-50). A fair test set, from [Samples and populations](https://learning.nextia-ai.com/courses/math/m05/samples-and-populations/#worked-example-mei-and-priya-check-the-300-test-tickets). |
| **Tested** | Python 3.14.6 (standard library only), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

The [lesson page](https://learning.nextia-ai.com/courses/math/m05/compare-alternatives/) has the full explanations, the charts, the explorers and the knowledge checks that count toward your certificate. Read a section there, then check its numbers here.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, replace each `...` with your calculation, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C03/M05-L03-compare-alternatives/compare-alternatives-solution.ipynb).

## Setup: the helpers

The next cell defines three small functions that the notebook uses. You do not need to read them, but you can.

- `expect(what, yours, expected)` compares your number with the lesson's number. It says "Check passed." or if yours is too high or too low. It does not show the answer.
- `table(headers, rows)` prints a small table with aligned columns.
- `bars(labels, counts)` prints a text bar chart: one row of `█` for each label.

Run the cell. You should see `Ready.`

In [ ]:
# Helpers: check an answer without showing it, print a table, draw text bars.
import math, random, statistics

def expect(what, yours, expected, tol=0.01):
    """Compare your value with the expected one, to within `tol` for numbers."""
    if yours is ...:
        print(f"Not yet: replace ... with your calculation for {what}, then run the cell again.")
        return
    if isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        if abs(yours - expected) <= tol:
            print(f"Check passed: {what} is {yours:g}.")
        else:
            side = "too high" if yours > expected else "too low"
            print(f"Not yet: your {what}, {yours:g}, is {side}. Look at the step before it again.")
    elif yours == expected:
        print(f"Check passed: {what} is {yours}.")
    else:
        print(f"Not yet: your {what} is {yours!r}. That is not the expected answer.")

def table(headers, rows, digits=2):
    """Print rows as a table. Floats are rounded to `digits` decimals."""
    cell = lambda v: f"{v:,.{digits}f}" if isinstance(v, float) else str(v)
    text = [[cell(v) for v in row] for row in rows]
    widths = [max(len(str(h)), *(len(r[i]) for r in text)) for i, h in enumerate(headers)]
    print("  ".join(str(h).rjust(w) for h, w in zip(headers, widths)))
    for r in text:
        print("  ".join(v.rjust(w) for v, w in zip(r, widths)))

def bars(labels, counts, width=40):
    """Print a text bar chart: the longest bar is `width` characters."""
    top = max(counts) or 1
    size = max(len(str(label)) for label in labels)
    for label, n in zip(labels, counts):
        print(f"{str(label).rjust(size)} | {'█' * round(n / top * width)} {n}")

print("Ready.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Observed effect: one difference in four units

Model B is correct on 276 of the 300 test tickets, model A on 273. An **effect size** is the size of a difference, in a unit that people understand.

> **Predict.** "1 point" and "11% fewer errors": can both describe the same result? Then run the cell.

In [ ]:
n = 300
correct_a, correct_b = 273, 276
errors_a, errors_b = n - correct_a, n - correct_b
per_week = 1200 / n                     # Larkfield gets about 1,200 tickets a week

table(["Unit", "Model A", "Model B", "Difference"], [
    ["Accuracy (%)", 100 * correct_a / n, 100 * correct_b / n, 100 * (correct_b - correct_a) / n],
    ["Correct tickets of 300", correct_a, correct_b, correct_b - correct_a],
    ["Wrong tickets of 300", errors_a, errors_b, errors_b - errors_a],
    ["Fewer errors (% of A's errors)", "", "", 100 * (errors_a - errors_b) / errors_a],
    ["Wrong-team tickets per week", errors_a * per_week, errors_b * per_week, (errors_b - errors_a) * per_week],
], digits=1)

> **Check.** You should see +1.0 point, 3 tickets, 3 fewer errors, 11.1% fewer errors, and 108 against 96 wrong-team tickets a week (12 fewer). All rows describe the same 3 tickets. Report the absolute numbers first.

## 2. Look at the tickets where the models disagree

The next cell writes the results of both models on the same 300 tickets: `1` is correct and `0` is wrong. The tickets are in four groups, as in the lesson's table. Then it counts each combination.

Run the cell. You should see the lesson's table: 261, 12, 15 and 12.

In [ ]:
# One entry per ticket: 1 = correct, 0 = wrong. The same 300 tickets for both models.
a = [1] * 261 + [1] * 12 + [0] * 15 + [0] * 12
b = [1] * 261 + [0] * 12 + [1] * 15 + [0] * 12

both_right = sum(1 for x, y in zip(a, b) if x == 1 and y == 1)
only_a = sum(1 for x, y in zip(a, b) if x == 1 and y == 0)
only_b = sum(1 for x, y in zip(a, b) if x == 0 and y == 1)
both_wrong = sum(1 for x, y in zip(a, b) if x == 0 and y == 0)
table(["", "B correct", "B wrong", "Total"], [
    ["A correct", both_right, only_a, both_right + only_a],
    ["A wrong", only_b, both_wrong, only_b + both_wrong],
    ["Total", both_right + only_b, only_a + both_wrong, len(a)],
])
print()
print("disagreements:", only_a + only_b, "| B wins", only_b, "| A wins", only_a, "| difference", only_b - only_a)

> **Check.** On 273 tickets the models agree (261 both right, 12 both wrong). Those tickets say nothing about which model is better. The evidence is in the 27 disagreements: B wins 15, A wins 12, and 15 − 12 = 3 is the 3-ticket difference from section 1.

## 3. If A and B were equally good: flip a coin

The **null hypothesis** says that there is no real difference. Then each of the 27 disagreements could go to A or to B with the same chance, like a coin. The next cell does what the lesson's explorer does: it flips 27 coins, records "B wins minus A wins", and repeats this 10,000 times.

> **Predict.** In how many of 100 shuffles is the difference 3 or more, in either direction: about 5, or many more? Then run the cell.

In [ ]:
def shuffle_difference(disagreements):
    """Flip one coin per disagreement; return B wins minus A wins."""
    b_wins = sum(random.random() < 0.5 for _ in range(disagreements))
    return b_wins - (disagreements - b_wins)

disagreements = 27     # B wins 15, A wins 12
real = 3               # the real difference: 15 - 12

random.seed(11)
diffs = [shuffle_difference(disagreements) for _ in range(10_000)]
extreme = len([d for d in diffs if abs(d) >= real])
print(f"{extreme} of {len(diffs)} shuffles ({extreme / len(diffs):.1%}) give a difference of {real} or more")

> **Check.** You should see `7093 of 10000 shuffles (70.9%)`. A split like 15 to 12 is normal when the two models are equally good.

The next cell draws the 10,000 differences as text bars. The differences are odd numbers, because 27 is odd. Run it. You should see the tallest bars at −1 and +1, and almost all differences between −13 and +13. The chart stops at ±15; only 16 shuffles are further out.

In [ ]:
values = list(range(-15, 16, 2))
counts = [diffs.count(v) for v in values]
bars([f"{v:+d}" for v in values], counts, width=30)

## 4. The p-value, exactly

The **p-value** is the share of chance-only results that are at least as extreme as the real result. You can calculate it exactly with the binomial distribution: the chance of exactly k heads in 27 fair coin flips is C(27, k) ÷ 2²⁷. `math.comb(27, k)` gives C(27, k).

> **Predict.** Will the exact p-value be close to the share of shuffles from section 3? Then run the cell.

In [ ]:
def exact_p(b_wins, a_wins):
    """Two-sided p-value: the chance of a split at least as uneven, if each disagreement is a fair coin."""
    total = b_wins + a_wins
    real = abs(b_wins - a_wins)
    return sum(math.comb(total, k) for k in range(total + 1) if abs(2 * k - total) >= real) / 2 ** total

print(f"exact p-value: {exact_p(15, 12):.2f}")

> **Check.** You should see `0.70`, the same as the explorer's exact p-value. The shuffles gave 0.709: counting gets close to the exact value.

Remember what p = 0.70 is **not**. It is not the probability that B is better, it is not the size of the effect, and it does not prove that the models are equal. In Python, this paired test is called **McNemar's test** (for example in `statsmodels`).

## 5. An interval for the difference

An interval also shows how large the difference could be. For two models on the same tickets, it uses the disagreement counts b (B right, A wrong) and c (A right, B wrong). The next cell follows the lesson's steps.

Run the cell. You should see the lesson's steps: a difference of +1.0 point, √26.97 = 5.19, an SE of 1.73 points, a half-width of 3.4 points and the interval **−2.4 to +4.4 points**.

In [ ]:
def paired_interval(b, c, n):
    """The difference (b − c) ÷ n, its SE and its 95% interval, in percentage points."""
    diff = (b - c) / n
    se = math.sqrt(b + c - (b - c) ** 2 / n) / n
    half = 1.96 * se
    return 100 * diff, 100 * se, 100 * (diff - half), 100 * (diff + half)

b, c, n = 15, 12, 300
table(["Step", "Value"], [
    ["Difference (b − c) ÷ n, in points", 100 * (b - c) / n],
    ["b + c", b + c],
    ["(b − c)² ÷ n", (b - c) ** 2 / n],
    ["√(b + c − (b − c)² ÷ n)", math.sqrt(b + c - (b - c) ** 2 / n)],
])
diff, se, low, high = paired_interval(b, c, n)
print()
print(f"SE {se:.2f} points, half-width {1.96 * se:.1f} points, 95% interval {low:+.1f} to {high:+.1f} points")

> **Check.** The interval contains zero. The data fits "B is 4 points better", "no difference" and "B is 2 points worse".

The lesson's chart also shows a wider grey interval: the same difference if the two models had been tested on **different** tickets. Then the SE combines the two separate SEs from Lesson 2. Run the cell. You should see about −3.5 to +5.5 points: ±4.5 points instead of ±3.4.

In [ ]:
se_a = math.sqrt(0.91 * 0.09 / 300)
se_b = math.sqrt(0.92 * 0.08 / 300)
se_unpaired = math.sqrt(se_a ** 2 + se_b ** 2)
half = 100 * 1.96 * se_unpaired
print(f"different tickets: ±{half:.1f} points, from {1.0 - half:+.1f} to {1.0 + half:+.1f} points")

## 6. Practical significance: does it matter?

Suppose B is really 1 point better. Omar asks for the costs, as in the lesson's table.

> **Your turn.** Replace each `...` with a calculation, then run the check cell.
>
> - `minutes_saved`: the agent minutes saved per week. B makes about 12 fewer wrong-team tickets a week, and each one costs about 5 minutes to move.
> - `weeks_to_pay_back`: the switch costs about 30 hours. How many weeks until the saved minutes pay it back?

In [ ]:
fewer_per_week = 12
minutes_per_ticket = 5
switch_hours = 30

minutes_saved = fewer_per_week * minutes_per_ticket
weeks_to_pay_back = switch_hours * 60 / minutes_saved

Run the check cell.

In [ ]:
expect("the minutes saved per week", minutes_saved, 60)
expect("the weeks to pay back", weeks_to_pay_back, 30)

The gain is worth having only **if** it is real, and the test cannot show that. A difference can also be statistically significant without practical value: with a very large test set, a tiny gain gets a tiny p-value.

## 7. Failure case: a before-and-after comparison

Tomás switched the live system to model B in week 41. Grace's report shows the weekly share of re-routed tickets. A delivery outage ran in weeks 38 to 40 and ended in week 41.

> **Predict.** Tomás compares week 40 with week 41. Grace compares the weeks before the outage with the weeks after the switch. Which comparison gives the larger fall? Then run the cell.

In [ ]:
rerouted = {35: 9.1, 36: 9.4, 37: 9.2, 38: 11.8, 39: 12.4, 40: 11.6,   # model A
            41: 8.1, 42: 8.0, 43: 8.4, 44: 8.2}                        # model B
bars([f"W{week}" for week in rerouted], list(rerouted.values()), width=40)

before_outage = statistics.mean([rerouted[w] for w in [35, 36, 37]])
after_switch = statistics.mean([rerouted[w] for w in [41, 42, 43, 44]])
print()
print(f"Tomás: week 40 to week 41: {rerouted[40]} to {rerouted[41]} ({(rerouted[41] - rerouted[40]) / rerouted[40]:.0%})")
print(f"Grace: weeks 35-37 to weeks 41-44: {before_outage:.1f} to {after_switch:.1f} ({after_switch - before_outage:+.1f} points)")

> **Check.** You should see Tomás's fall from 11.6% to 8.1% (−30%), and Grace's from 9.2% to 8.2% (−1.1 points). Most of Tomás's fall came from the end of the outage. Even Grace's 1 point could come from other changes. The fall is **correlated** with the switch; only an **A/B test**, with random assignment of tickets to A or B in the same weeks, shows the **cause**.

## 8. Module practice: model C

### Worked example: the five checks for A and B

Priya's five checks use the numbers that you calculated above. Run the cell to see them together.

In [ ]:
diff, se, low, high = paired_interval(15, 12, 300)
table(["Check", "Result for A vs B"], [
    ["1. Fair, same test set?", "yes: 300 random tickets, split by customer"],
    ["2. Observed effect", f"{diff:+.1f} point (276 vs 273 of 300)"],
    ["3. Uncertainty", f"95% interval {low:+.1f} to {high:+.1f} points, p = {exact_p(15, 12):.2f}"],
    ["4. Practical value", "about 60 minutes a week, if real; the switch costs 30 hours"],
    ["5. Cause", "the same tickets: the models or chance; live effect needs an A/B test"],
])

### Your task: model C

Mei builds a new test set: 1,000 random tickets from the first two weeks of March, split by customer. Model A is right on 910, model C on 929. They disagree on 63 tickets: C is right and A wrong on b = 41, A is right and C wrong on c = 22.

Tomás writes: "Model C is clearly better than A, and it will cut re-routed tickets by a third."

> **Your turn.** Replace each `...` with a calculation. You can use the steps of section 5, or the functions `paired_interval` and `exact_p`. Give the effect, the SE and the interval in percentage points. Then run the check cell. Write your answers to steps 2, 6 and 7 of the lesson's task in `c03-notes.md`.

In [ ]:
b_c, c_c, n_c = 41, 22, 1000

effect_points = 100 * (b_c - c_c) / n_c
fewer_errors = 100 * (b_c - c_c) / 90
se_points = 100 * math.sqrt(b_c + c_c - (b_c - c_c) ** 2 / n_c) / n_c
low_c = effect_points - 1.96 * se_points
high_c = effect_points + 1.96 * se_points
p_value = exact_p(b_c, c_c)

Run the check cell.

In [ ]:
expect("the effect", effect_points, 1.9)
expect("the fewer errors (%)", fewer_errors, 21.1, tol=0.05)
expect("the SE", se_points, 0.79, tol=0.01)
expect("the low end", low_c, 0.35, tol=0.02)
expect("the high end", high_c, 3.45, tol=0.02)
expect("the p-value", p_value, 0.023, tol=0.001)

> **Your turn: change one thing.** In the first cell of section 3, set `disagreements = 63` and `real = 19` (41 − 22 = 19), and run it. You should see `243 of 10000 shuffles (2.4%)`, close to the exact p-value of 0.023. A split of 41 to 22 is rare by chance alone.

Compare your conclusion for Omar with the model answer on the lesson page: is "clearly better" supported? Is "a third fewer re-routes"?

## Check your understanding and recap

Answer the **knowledge checks** and the **module check** on the [lesson page](https://learning.nextia-ai.com/courses/math/m05/compare-alternatives/#module-check). They count toward your certificate when you are signed in and enrolled. The notebook does not have them.

In this notebook, model B's +1.0 point was 3 tickets of 300. The evidence was in the 27 disagreements, and coin flips gave a split at least as uneven as 15 to 12 in about 70% of shuffles (exact p = 0.70). The 95% interval for the difference, −2.4 to +4.4 points, contains zero. The fall in re-routes after week 41 was mostly the end of the outage: a correlation, not a cause. For model C, the interval +0.35 to +3.45 points and p = 0.023 support "probably a little better", not "a third fewer re-routes".

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Classification trade-offs](https://learning.nextia-ai.com/courses/math/m06/classification-trade-offs/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/math/m05/compare-alternatives/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield tickets are fictional and have no real people.